In [8]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity

ING_META_PATH = "ingredient_meta.csv"
ING_EMB_PATH  = "ingredient_embeddings.npy"

성분 메타 & 임베딩 로드

In [9]:
ingredient_meta = pd.read_csv(ING_META_PATH)
ingredient_embeddings = np.load(ING_EMB_PATH)

assert "ingredient_name" in ingredient_meta.columns
assert len(ingredient_meta) == ingredient_embeddings.shape[0]

EMBEDDING_DIM = ingredient_embeddings.shape[1]

print("성분 수:", len(ingredient_meta))
print("임베딩 차원:", EMBEDDING_DIM)

성분 수: 3149
임베딩 차원: 768


성분명 정규화 함수

In [10]:
def norm_ing(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[\[\]\(\)\{\}]", " ", s)
    s = re.sub(r"[^0-9a-z가-힣\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

정규화 성분 dict 생성

In [11]:
ingredient_meta["ingredient_name_norm"] = ingredient_meta["ingredient_name"].map(norm_ing)

ingredient_embedding_dict = {}
for idx, row in ingredient_meta.iterrows():
    key = row["ingredient_name_norm"]
    if key and key not in ingredient_embedding_dict:
        ingredient_embedding_dict[key] = ingredient_embeddings[idx]

print("정규화 성분 dict 크기:", len(ingredient_embedding_dict))

정규화 성분 dict 크기: 3048


특정 키워드가 실제로 어떤 성분을 잡는지 확인

In [12]:
for k in ingredient_embedding_dict.keys():
    if "병풀" in k or "인삼" in k or "centella" in k or "ginseng" in k:
        print(k)

2-헥산다이올 펜틸렌글라이콜 카프릴릴글라이콜 캐모마일꽃추출물 병풀추출물 마데카소사이드
병풀꽃 잎 줄기추출물
병풀꽃 잎 줄기추출물 플루이드 정제수
병풀꽃 잎 줄기추출물 25 ppm
병풀꽃 잎 줄기추출물 3 125 ppm
병풀꽃 잎 줄기추출물 3 125ppm
병풀잎추출물
병풀잎추출물 5
병풀잎추출물분말
병풀추출물
병풀추출물 10
병풀추출물 10ppm
인삼가루
인삼꽃추출물
인삼뿌리세포추출물
인삼뿌리프로토플라스트
인삼수
인삼씨오일
인삼열매농축액 국산
인삼열매추출물
인삼잎 줄기추출물
인삼전초추출물
인삼추출물
제삼인삼칼슘
하이드롤라이즈드인삼사포닌


상위 키워드 → 관련 성분 전부 매칭 함수

In [13]:
def find_group_embeddings(keyword, emb_dict):
    key = norm_ing(keyword)
    vecs = []
    matched_keys = []

    for k, v in emb_dict.items():
        if key in k:
            vecs.append(v)
            matched_keys.append(k)

    return vecs, matched_keys

성분 그룹 평균 벡터 생성

In [14]:
def build_group_vector(keywords, emb_dict, dim):
    all_vecs = []
    match_log = {}

    for kw in keywords:
        vecs, keys = find_group_embeddings(kw, emb_dict)
        match_log[kw] = keys
        all_vecs.extend(vecs)

    if all_vecs:
        return np.mean(np.stack(all_vecs), axis=0), match_log
    else:
        return np.zeros(dim, dtype=np.float32), match_log

페르소나 성분 선호 정의

In [15]:
persona_ingredient_preference = {
    "persona_1": ["히알루론산", "시카"],
    "persona_2": ["나이아신아마이드"],
    "persona_3": ["병풀", "인삼"]
}

페르소나 성분 벡터 생성 + 매칭 로그

In [16]:
persona_ingredient_vector = {}
persona_match_log = {}

for pid, keywords in persona_ingredient_preference.items():
    vec, matched = build_group_vector(
        keywords,
        ingredient_embedding_dict,
        EMBEDDING_DIM
    )
    persona_ingredient_vector[pid] = vec
    persona_match_log[pid] = matched

for pid, vec in persona_ingredient_vector.items():
    print(pid, "ingredient vec norm:", np.linalg.norm(vec))

persona_1 ingredient vec norm: 20.516138
persona_2 ingredient vec norm: 19.073086
persona_3 ingredient vec norm: 20.689104


실제로 묶인 성분들 출력

In [17]:
for pid, matched in persona_match_log.items():
    print(f"\n[{pid}]")
    for kw, keys in matched.items():
        print(f"  - {kw}: {len(keys)}개")
        for k in keys[:5]:
            print("    ", k)


[persona_1]
  - 히알루론산: 3개
     히알루론산
     히알루론산 난류 가금류
     히알루론산혼합제제 덱스트린
  - 시카: 7개
     리포좀 시카 b5 판토텐산 밀크세라마이드 버터유분말 독일산
     트라이에톡시카프릴릴실레인
     트라이에톡시카프릴릴실레인 12 탤크
     트라이에톡시카프릴릴실레인 3 탤크
     트라이에톡시카프릴릴실레인 danced 탤크

[persona_2]
  - 나이아신아마이드: 5개
     나이아신아마이드
     나이아신아마이드 0 11
     나이아신아마이드 20
     나이아신아마이드 40
     나이아신아마이드 50

[persona_3]
  - 병풀: 12개
     2-헥산다이올 펜틸렌글라이콜 카프릴릴글라이콜 캐모마일꽃추출물 병풀추출물 마데카소사이드
     병풀꽃 잎 줄기추출물
     병풀꽃 잎 줄기추출물 플루이드 정제수
     병풀꽃 잎 줄기추출물 25 ppm
     병풀꽃 잎 줄기추출물 3 125 ppm
  - 인삼: 13개
     인삼가루
     인삼꽃추출물
     인삼뿌리세포추출물
     인삼뿌리프로토플라스트
     인삼수


페르소나 톤 벡터

In [18]:
persona_tone = {
    "persona_1": np.zeros(EMBEDDING_DIM, dtype=np.float32),
    "persona_2": np.zeros(EMBEDDING_DIM, dtype=np.float32),
    "persona_3": np.zeros(EMBEDDING_DIM, dtype=np.float32),
}

리스크 / 가격 수치 벡터

In [19]:
# [민감도, 트러블, 향 민감도, 가격 민감도]
persona_risk_price_vector = {
    "persona_1": [1.0, 0.9, 0.8, 0.3],
    "persona_2": [0.3, 0.2, 0.2, 0.9],
    "persona_3": [0.4, 0.3, 0.1, 0.2],
}

페르소나 최종 벡터 concat

In [20]:
persona_ids = sorted(
    set(persona_tone)
    & set(persona_ingredient_vector)
    & set(persona_risk_price_vector)
)

persona_final_vector = {}

for pid in persona_ids:
    final_vec = np.concatenate([
        persona_tone[pid],
        persona_ingredient_vector[pid],
        np.array(persona_risk_price_vector[pid], dtype=np.float32)
    ])
    persona_final_vector[pid] = final_vec

print("최종 벡터 차원:", len(next(iter(persona_final_vector.values()))))

최종 벡터 차원: 1540


CSV 저장

In [21]:
rows = []
for pid in persona_ids:
    rows.append({
        "persona_id": pid,
        "tone_vector": persona_tone[pid].tolist(),
        "ingredient_vector": persona_ingredient_vector[pid].tolist(),
        "risk_price_vector": persona_risk_price_vector[pid],
        "final_vector": persona_final_vector[pid].tolist()
    })

persona_df = pd.DataFrame(rows)
persona_df.to_csv("persona_vectors.csv", index=False)

persona_df.head()

,persona_id,tone_vector,ingredient_vector,risk_price_vector,final_vector
0,persona_1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.01521330513060093, 0.3542287051677704, -3.9...","[1.0, 0.9, 0.8, 0.3]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,persona_2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.013257542625069618, 0.014659399166703224, ...","[0.3, 0.2, 0.2, 0.9]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,persona_3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.05767958611249924, 0.3115178346633911, -3.7...","[0.4, 0.3, 0.1, 0.2]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


페르소나 간 cosine similarity

In [22]:
vecs = np.stack([persona_final_vector[p] for p in persona_ids])
cosine_similarity(vecs)

array([[1.0000001 , 0.8850165 , 0.9704628 ],
       [0.8850165 , 0.9999999 , 0.90628296],
       [0.9704628 , 0.90628296, 0.99999976]], dtype=float32)

In [23]:
persona_ids_sorted = sorted(persona_final_vector.keys())
persona_matrix = np.stack([persona_final_vector[pid] for pid in persona_ids_sorted])

np.save("persona_vectors.npy", persona_matrix)

In [24]:
pd.DataFrame({
    "persona_id": persona_ids_sorted
}).to_csv("persona_meta.csv", index=False)